# Building a LangChain SQL Demo in Jupyter Notebook

## Step 1: Installation


In [2]:
!pip install langchain langchain-community sqlalchemy pandas ollama

     ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 MB 660.6 kB/s eta 0:00:04
     ---------------------------------------- 0.0/2.5 MB 660.6 kB/s eta 0:00:04
     - -------------------------------------- 0.1/2.5 MB 491.5 kB/s eta 0:00:06
     - -------------------------------------- 0.1/2.5 MB 590.8 kB/s eta 0:00:05
     --- ------------------------------------ 0.2/2.5 MB 888.4 kB/s eta 0:00:03
     --- ------------------------------------ 0.2/2.5 MB 958.6 kB/s eta 0:00:03
     ------ --------------------------------- 0.4/2.5 MB 1.3 MB/s eta 0:00:02
     ------- -------------------------------- 0.5/2.5 MB 1.5 MB/s eta 0:00:02
     ------------- -------------------------- 0.8/2.5 MB 1.9 MB/s eta 0:00:01
     ---------------- ----------------------- 1.0/2.5 MB 2.2 MB/s eta 0:00:01
     ------------------ --------------------- 1.2/2.5 MB 2.3 MB/s eta 0:00:01
     ------------------------- -------------- 1.6/2.5 MB 3.


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 2: Import Libraries and Set API Key


In [1]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine
from langchain.utilities import SQLDatabase
from langchain.llms import Ollama
from langchain_experimental.sql import SQLDatabaseChain
from langchain.agents import create_sql_agent
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import ollama  # For direct interaction with Ollama

# Step 3: Create Sample Database and Data

In [2]:
# Create a SQLite database in memory (you can change to a file path if you want to persist)
conn = sqlite3.connect('company.db')
cursor = conn.cursor()

# Create departments table
cursor.execute('''
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    location TEXT
)
''')

# Create employees table
cursor.execute('''
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    salary REAL,
    department_id INTEGER,
    hire_date TEXT,
    FOREIGN KEY (department_id) REFERENCES departments (id)
)
''')

# Insert sample data into departments
departments_data = [
    (1, 'Engineering', 'New York'),
    (2, 'Sales', 'Chicago'),
    (3, 'Marketing', 'San Francisco'),
    (4, 'HR', 'Boston')
]
cursor.executemany('INSERT INTO departments VALUES (?, ?, ?)', departments_data)

# Insert sample data into employees
employees_data = [
    (1, 'John Doe', 75000, 1, '2020-01-15'),
    (2, 'Jane Smith', 85000, 1, '2019-03-23'),
    (3, 'Robert Johnson', 65000, 2, '2021-07-01'),
    (4, 'Emily Davis', 95000, 2, '2018-05-12'),
    (5, 'Michael Brown', 70000, 3, '2022-02-28'),
    (6, 'Sarah Wilson', 80000, 3, '2020-11-05'),
    (7, 'David Thompson', 90000, 1, '2017-09-19'),
    (8, 'Jessica Garcia', 60000, 4, '2021-04-15'),
    (9, 'Christopher Martinez', 72000, 2, '2019-08-22'),
    (10, 'Amanda Rodriguez', 88000, 1, '2018-12-10')
]
cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?, ?)', employees_data)

# Commit changes and close connection
conn.commit()

# Let's verify our data
print("Departments:")
print(pd.read_sql_query("SELECT * FROM departments", conn))
print("\nEmployees:")
print(pd.read_sql_query("SELECT * FROM employees", conn))

# Close the connection
conn.close()

OperationalError: table departments already exists

# Step 4: Set Up Database Connection for LangChain


In [3]:
# Create a SQLAlchemy engine
engine = create_engine('sqlite:///company.db')

# Create the SQLDatabase object for LangChain
db = SQLDatabase(engine)

# Let's see what tables are available
print(db.get_usable_table_names())

['departments', 'employees']


# Step 5: Initialize the Language Model


In [ ]:
# Initialize the Ollama model
llm = Ollama(model="llama3.3", temperature=0.1)

# Test the model with a simple prompt
response = llm.invoke("What is the capital of France?")
print("Model test response:", response)

# Step 6: Create and Use SQLDatabaseChain


In [11]:
from langchain.chains import create_sql_query_chain
from langchain_experimental.sql import SQLDatabaseChain
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Step 1: Create a chain to generate SQL
query_chain = create_sql_query_chain(llm, db)

# Step 2: Generate SQL from a question
question = "give me the list of employees who were hired after 2020 and give me their salaries and their department name"
sql_response = query_chain.invoke({"question": question})

# Extract only the SQL query from the response
import re
match = re.search(r"SQLQuery:\s*(SELECT[\s\S]+)", sql_response)
if match:
	clean_sql = match.group(1).strip()
else:
	raise ValueError("Could not extract SQL query from response")

# Step 3: Execute query
sql_result = db.run(clean_sql)

# Step 4: Analyze results directly
analysis_prompt = PromptTemplate.from_template("""
You are a data analyst. Here is the SQL result, Please tell me the chances of success for each employee:

{data}

Answer the user's question: {question}
""")
analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)

analysis = analysis_chain.run(data=sql_result, question=question)



ValueError: Could not extract SQL query from response

In [12]:
# print("SQL Query!!!!!:", sql)
print("Result!!!!!!!!:", sql_result)
print("Analysis!!!!!!:", analysis)


Result!!!!!!!!: [('Robert Johnson', 65000.0, 'department_name'), ('Michael Brown', 70000.0, 'department_name'), ('Jessica Garcia', 60000.0, 'department_name')]
Analysis!!!!!!: Based on the provided SQL result, I can tell you that there is no direct information about the date of hire for each employee. However, we can make an educated guess based on the salary data.

Assuming a general salary scale, here's my analysis:

- Robert Johnson has a salary of $65,000, which is relatively low compared to other salaries in the list.
- Michael Brown has a salary of $70,000, which is higher than Robert Johnson's but lower than Jessica Garcia's.
- Jessica Garcia has a salary of $60,000, which is lower than Michael Brown's.

Based on this analysis, it seems that Michael Brown was hired after 2020 and likely before the other two employees. However, without more information about the hiring dates or salaries over time, we can't make any definitive conclusions.

As for your second question, to get the 

In [13]:
sql_response

'To answer this question, we need to query the `employees` table for employees with a hire date greater than 2020, along with their salary and department name. We also need to join the `departments` table to get the department names.\n\n```sql\nSELECT \n    "name", \n    "salary", \n    "department_name"\nFROM \n    employees\nINNER JOIN \n    departments ON employees."department_id" = departments.id\nWHERE \n    STRFTIME(\'%Y\', "hire_date") > \'2020\'\nORDER BY \n    "salary" DESC;\n```'